In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresConnector") \
    .config("spark.jars","/usr/lib/spark/jars/postgresql-42.7.4.jar")\
    .getOrCreate()



In [ ]:
employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

# Question 52: List employees who have worked in Development but never in Research
# Concept: EXCEPT / Left Anti Join

# 1. Create aliases for clarity (same as Question51)
e = employee_df.alias("e")
d = department_df.alias("d")
de = department_employee_df.alias("de")

# 2. Get employees who worked in Development
dev = e.join(de, e["id"] == de["employee_id"]) \
        .join(d, de["department_id"] == d["id"]) \
        .filter(col("d.dept_name") == "Development") \
        .select(col("e.id")).distinct()

# 3. Get employees who worked in Research
res = e.join(de, e["id"] == de["employee_id"]) \
        .join(d, de["department_id"] == d["id"]) \
        .filter(col("d.dept_name") == "Research") \
        .select(col("e.id")).distinct()

# 4. Left Anti Join: Get Dev employees who are NOT in Research
dev.join(res, "id", "left_anti").join(e, "id").show()
